# Персональные PDF-отчёты для преподавателей

Генерирует один PDF (1 лист А4 горизонтально) на каждого преподавателя.

**Структура отчёта:**
- **Раздел А** — 8 общих метрик по программам преподавателя (из общего опроса)
- **Раздел Б** — персональные оценки: среднее, медиана, количество ответов, программ
- **Правая панель** — гистограмма распределения оценок и таблица по программам

**Данные:** `combined_general_agg.csv` и `data/processed/combined_teachers_agg.csv`  
**Выходные файлы:** `output/teacher_pdfs/<Имя>.pdf`  
**Зависимости:** `pip install pandas numpy matplotlib`

---

Запускайте ячейки последовательно. В последней ячейке укажите фильтры (`SEL_SCHOOLS`, `SEL_PROGRAMS`, `ONLY_ONE`) прямо в коде перед запуском.

In [107]:
import os, re, warnings, logging
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
from matplotlib.backends.backend_pdf import PdfPages

warnings.filterwarnings("ignore")
logging.getLogger("fontTools").setLevel(logging.ERROR)   # подавить "feat NOT subset"
matplotlib.rcParams["pdf.fonttype"] = 42                 # embed fonts (no Type 3)
print("Библиотеки загружены.")

Библиотеки загружены.


In [108]:
# ── Пути к файлам ──────────────────────────────────────────────────────────
BASE_DIR     = Path("..")
GENERAL_CSV  = BASE_DIR / "combined_general_agg.csv"
TEACHERS_CSV = BASE_DIR / "data/processed/combined_teachers_agg.csv"
OUTPUT_DIR   = BASE_DIR / "output/teacher_pdfs"

# ── Шрифт ──────────────────────────────────────────────────────────────────
_font_dirs = [
    Path.home() / "Library/Fonts",
    Path("/Library/Fonts"),
    Path("/System/Library/Fonts"),
]
for _d in _font_dirs:
    for _f in _d.glob("*.[ot]tf"):
        if "Univers" in _f.stem:
            fm.fontManager.addfont(str(_f))

_avail_fonts = {f.name for f in fm.fontManager.ttflist}
FONT_FAMILY  = next(
    (f for f in ["Univers LT CYR", "UniversLTCYR", "Univers"] if f in _avail_fonts),
    "sans-serif",
)
print(f"Шрифт: {FONT_FAMILY}")

# ── Палитра: строго чёрно-белая ────────────────────────────────────────────
BLACK      = "#111111"
GRAY_DARK  = "#3D3D3D"
GRAY_MID   = "#888888"
GRAY_LIGHT = "#DEDEDE"
BG_CARD    = "#F5F5F5"
WHITE      = "#FFFFFF"

plt.rcParams.update({
    "font.family":        FONT_FAMILY,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.left":   False,
    "axes.spines.bottom": False,
    "axes.facecolor":     WHITE,
    "figure.facecolor":   WHITE,
    "text.color":         BLACK,
})

# ── Метрики раздела А (7 штук из общего опроса) ────────────────────────────
# Первая карточка в разделе А — средний балл преподавателя (добавляется динамически)
# (колонка_в_csv, полное_название, максимум_шкалы)
SECTION_A_METRICS = [
    ("satisf_overall",         "Удовлетворённость\nобразовательным процессом",    4),
    ("satisf_teachers",        "Удовлетворённость\nпреподавателями",               4),
    ("assess_criteria_timely", "Критерии оценивания\nсообщены вовремя",            5),
    ("assess_order_clear",     "Порядок сдачи работ\nпонятен студентам",           5),
    ("assess_consistent",      "Оценивание соответствует\nобъявленным критериям",  5),
    ("fac_support_score",      "Поддержка со стороны\nпреподавателей",             4),
    ("fac_clarity_score",      "Понятность объяснений\nпреподавателей",            4),
]

print(f"Метрики раздела А:  {len(SECTION_A_METRICS)} + 1 карточка среднего балла")
print(f"Папка вывода:       {OUTPUT_DIR}")

Шрифт: Univers LT CYR
Метрики раздела А:  7 + 1 карточка среднего балла
Папка вывода:       ../output/teacher_pdfs


In [109]:
print("Загрузка данных...")

general_df  = pd.read_csv(GENERAL_CSV)
teachers_df = pd.read_csv(TEACHERS_CSV)
teachers_df["rating"] = pd.to_numeric(teachers_df["rating"], errors="coerce")

print(f"  Общий опрос:     {len(general_df):,} строк")
print(f"  Оценки учителей: {len(teachers_df):,} строк, "
      f"{teachers_df['teacher'].nunique()} преподавателей")

Загрузка данных...
  Общий опрос:     236 строк
  Оценки учителей: 2,321 строк, 204 преподавателей


In [110]:
# ── Вспомогательные функции ────────────────────────────────────────────────

def safe_filename(name: str) -> str:
    return re.sub(r'[^\w\s\-]', '', name).strip().replace(' ', '_')


def fmt_score(v, scale: int) -> str:
    """Переводит абсолютный балл в % от максимума шкалы."""
    if pd.isna(v):
        return "—"
    return f"{v / scale * 100:.0f}%"


def report_date_str() -> str:
    months = {
        1: "января", 2: "февраля", 3: "марта", 4: "апреля",
        5: "мая", 6: "июня", 7: "июля", 8: "августа",
        9: "сентября", 10: "октября", 11: "ноября", 12: "декабря",
    }
    d = datetime.now()
    return f"{d.day} {months[d.month]} {d.year}"


print("Вспомогательные функции определены.")

Вспомогательные функции определены.


In [111]:
# ── Функции отрисовки (строго чёрно-белые) ────────────────────────────────

def draw_header(ax, teacher_name: str, report_date: str, programs: list):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.add_patch(mpatches.FancyBboxPatch(
        (0, 0), 1, 1, boxstyle="square,pad=0",
        facecolor=BLACK, edgecolor="none",
        transform=ax.transAxes, clip_on=False,
    ))
    ax.text(0.025, 0.82, "РЕЗУЛЬТАТЫ СЕМЕСТРОВОГО ОПРОСА СТУДЕНТОВ · ДЕПАРТАМЕНТ АКАДЕМИЧЕСКОГО КАЧЕСТВА",
            color=WHITE, fontsize=6.5, alpha=0.55, fontweight="bold",
            transform=ax.transAxes)
    ax.text(0.025, 0.38, teacher_name,
            color=WHITE, fontsize=21, fontweight="black",
            transform=ax.transAxes)
    prog_str = "  ·  ".join(programs[:6])
    if len(programs) > 6:
        prog_str += f"  +{len(programs) - 6}"
    ax.text(0.025, 0.10, prog_str,
            color=WHITE, fontsize=6.5, alpha=0.50,
            transform=ax.transAxes)
    ax.text(0.975, 0.82, "СФОРМИРОВАНО АВТОМАТИЧЕСКИ",
            color=WHITE, fontsize=6.5, alpha=0.55, ha="right",
            transform=ax.transAxes)
    ax.text(0.975, 0.38, report_date,
            color=WHITE, fontsize=9, alpha=0.75, ha="right", fontweight="medium",
            transform=ax.transAxes)


def draw_section_label(ax, title: str):
    ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.add_patch(mpatches.Rectangle(
        (0, 0.1), 0.003, 0.80,
        facecolor=BLACK, edgecolor="none",
        transform=ax.transAxes,
    ))
    ax.text(0.012, 0.5, title.upper(),
            color=GRAY_MID, fontsize=6, va="center",
            fontweight="bold", transform=ax.transAxes)


def draw_metric_card(ax, title: str, value: str, accent: bool = False):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    if accent:
        ax.add_patch(mpatches.FancyBboxPatch(
            (0.04, 0.05), 0.92, 0.90,
            boxstyle="round,pad=0.02",
            facecolor=BLACK, edgecolor="none", linewidth=0,
        ))
        ax.text(0.5, 0.76, title,
                ha="center", va="center", fontsize=6,
                color=WHITE, alpha=0.60, multialignment="center")
        ax.text(0.5, 0.33, value,
                ha="center", va="center", fontsize=15,
                color=WHITE, fontweight="bold")
    else:
        ax.add_patch(mpatches.FancyBboxPatch(
            (0.04, 0.05), 0.92, 0.90,
            boxstyle="round,pad=0.02",
            facecolor=BG_CARD, edgecolor=GRAY_LIGHT, linewidth=0.7,
        ))
        ax.text(0.5, 0.76, title,
                ha="center", va="center", fontsize=6.5,
                color=GRAY_MID, multialignment="center")
        ax.text(0.5, 0.33, value,
                ha="center", va="center", fontsize=12,
                color=BLACK, fontweight="bold")


def draw_stat_card(ax, label: str, value: str, accent: bool = False):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    if accent:
        ax.add_patch(mpatches.FancyBboxPatch(
            (0.04, 0.05), 0.92, 0.90,
            boxstyle="round,pad=0.02",
            facecolor=BLACK, edgecolor=BLACK, linewidth=0,
        ))
        ax.text(0.5, 0.75, label,
                ha="center", va="center", fontsize=7,
                color=WHITE, alpha=0.65, multialignment="center")
        ax.text(0.5, 0.34, value,
                ha="center", va="center", fontsize=20,
                color=WHITE, fontweight="bold")
    else:
        ax.add_patch(mpatches.FancyBboxPatch(
            (0.04, 0.05), 0.92, 0.90,
            boxstyle="round,pad=0.02",
            facecolor=BG_CARD, edgecolor=GRAY_LIGHT, linewidth=0.7,
        ))
        ax.text(0.5, 0.75, label,
                ha="center", va="center", fontsize=7,
                color=GRAY_MID, multialignment="center")
        ax.text(0.5, 0.34, value,
                ha="center", va="center", fontsize=20,
                color=BLACK, fontweight="bold")


def draw_histogram(ax, ratings: pd.Series, t_mean: float):
    ax.spines["left"].set_visible(True)
    ax.spines["bottom"].set_visible(True)
    ax.spines["left"].set_color(GRAY_LIGHT)
    ax.spines["bottom"].set_color(GRAY_LIGHT)
    counts, _ = np.histogram(ratings.dropna(), bins=range(0, 12))
    bar_clrs = [BLACK if i == int(round(t_mean)) else GRAY_LIGHT for i in range(11)]
    ax.bar(range(0, 11), counts, color=bar_clrs, edgecolor=WHITE, linewidth=0.5)
    ax.set_xticks(range(0, 11))
    ax.tick_params(labelsize=6, colors=GRAY_MID)
    ax.set_xlabel("Оценка (0–10)", fontsize=6.5, color=GRAY_MID, labelpad=3)
    ax.set_ylabel("Ответов",       fontsize=6.5, color=GRAY_MID, labelpad=3)
    ax.set_title("Распределение оценок", fontsize=7.5,
                 fontweight="bold", color=BLACK, pad=5)
    ax.axvline(t_mean, color=GRAY_DARK, linewidth=1.2, linestyle="--", alpha=0.7)
    y_max = max(counts) if max(counts) > 0 else 1
    ax.text(min(t_mean + 0.25, 9.4), y_max * 0.88,
            f"μ = {t_mean:.1f}", fontsize=6.5, color=GRAY_DARK)
    ax.set_xlim(-0.6, 10.6)


def draw_programs_table(ax, rows: list):
    ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    if not rows:
        ax.text(0.5, 0.5, "Нет данных", ha="center", va="center",
                color=GRAY_MID, fontsize=8)
        return
    ax.set_title("По программам", fontsize=7.5, fontweight="bold",
                 color=BLACK, pad=4, loc="left")
    headers = ["Программа", "Курс", "N", "Ср."]
    col_x   = [0.00, 0.62, 0.76, 0.88]
    col_w   = [0.62, 0.14, 0.12, 0.12]
    y_start = 0.88
    row_h   = min(0.13, 0.85 / (len(rows) + 1))
    for h, x, w in zip(headers, col_x, col_w):
        ax.text(x + w / 2, y_start, h,
                ha="center", va="center",
                fontsize=6.5, fontweight="bold", color=GRAY_MID)
    ax.axhline(y_start - 0.05, color=GRAY_LIGHT, linewidth=0.8)
    for i, row in enumerate(rows):
        y = y_start - (i + 1) * row_h - 0.02
        if y < 0.02:
            break
        if i % 2 == 0:
            ax.add_patch(mpatches.Rectangle(
                (0, y - row_h * 0.45), 1, row_h * 0.9,
                facecolor=BG_CARD, edgecolor="none",
            ))
        vals = [
            str(row.get("program", ""))[:36],
            str(row.get("year", "")),
            str(row.get("n", "")),
            f"{row['mean']:.1f}" if row.get("mean") is not None else "—",
        ]
        for val, x, w in zip(vals, col_x, col_w):
            ax.text(x + w / 2, y, val,
                    ha="center", va="center",
                    fontsize=6, color=BLACK)


print("Функции отрисовки определены.")

Функции отрисовки определены.


In [112]:
# ── Основная функция генерации PDF ────────────────────────────────────────
# А4 горизонтально · чёрно-белый · constrained_layout
#
# СЕТКА: 6 строк × 12 колонок
#
#  строка 0            — шапка (cols 0–11, вся ширина)
#  строка 1            — метка раздела А (cols 0–7)   ┐
#  строки 2–3          — 8 карточек А   (cols 0–7)    │ правая панель:
#  строка 4            — метка раздела Б (cols 0–7)   │ cols 8–11,
#  строка 5            — 4 карточки Б   (cols 0–7)    ┘ строки 1–5
#
# height_ratios = [шапка, метка_А, карточки_А, карточки_А, метка_Б, карточки_Б]
# wspace/hspace — зазоры между ячейками (в долях от средней высоты/ширины)

def generate_pdf_for_teacher(
    teacher_name: str,
    teacher_df: pd.DataFrame,
    general_df: pd.DataFrame,
    output_path: str,
) -> None:

    # ── Данные ────────────────────────────────────────────────────────────
    ratings  = teacher_df["rating"].dropna()
    t_mean   = ratings.mean()
    t_n      = len(ratings)
    t_median = ratings.median() if t_n > 0 else np.nan

    program_rows = sorted([
        {"program": prog, "year": year,
         "n": len(grp), "mean": grp["rating"].mean()}
        for (prog, year), grp in teacher_df.groupby(["program", "year"])
    ], key=lambda r: r["n"], reverse=True)

    ctx = general_df[
        general_df["program"].isin(teacher_df["program"].unique()) &
        general_df["year"].isin(teacher_df["year"].unique())
    ]

    # Средний балл других преподавателей на тех же программах (для сравнения)
    peer_ctx = teachers_df[
        teachers_df["program"].isin(teacher_df["program"].unique()) &
        (teachers_df["teacher"] != teacher_name)
    ]
    peer_mean = peer_ctx["rating"].mean() if not peer_ctx.empty else np.nan

    # Раздел А: первая карточка — средний балл преподавателя (акцентная),
    # затем 7 метрик из общего опроса в процентах
    section_a_cards = [
        (
            "Средний балл\nдругих преподавателей\nна ваших программах",
            f"{peer_mean:.1f}" if not pd.isna(peer_mean) else "—",
            True,   # accent
        )
    ] + [
        (title, fmt_score(ctx[col].mean() if col in ctx.columns
                          and not ctx[col].dropna().empty else np.nan, scale), False)
        for col, title, scale in SECTION_A_METRICS
    ]

    b_cards = [
        ("Средний балл\n(шкала 0–10)",
         f"{t_mean:.1f}" if not pd.isna(t_mean) else "—", True),
        ("Медианный балл",
         f"{t_median:.1f}" if not pd.isna(t_median) else "—", False),
        ("Количество оценивших студентов", str(t_n), False),
        ("Количество программ и курсов",  str(len(program_rows)), False),
    ]

    date_str      = report_date_str()
    progs_display = list(dict.fromkeys(r["program"] for r in program_rows))

    with PdfPages(output_path) as pdf:
        fig = plt.figure(figsize=(11.69, 8.27), layout="constrained")
        fig.patch.set_facecolor(WHITE)

        gs = fig.add_gridspec(
            6, 12,
            height_ratios=[1.8, 0.25, 1.0, 1.0, 0.25, 1.1],
            #               шапка метА  картА картА метБ  картБ
            hspace=0.1,
            wspace=0.1,
        )

        # Строка 0: шапка
        draw_header(fig.add_subplot(gs[0, :]), teacher_name, date_str, progs_display)

        # Cols 8–11, строки 1–5: правая панель
        gs_right = gs[1:, 8:].subgridspec(3, 1, height_ratios=[0.25, 2, 1], hspace=0.1)
        if t_n > 0 and not pd.isna(t_mean):
            draw_histogram(fig.add_subplot(gs_right[1]), ratings, t_mean)
        else:
            ax_e = fig.add_subplot(gs_right[1])
            ax_e.axis("off")
            ax_e.text(0.5, 0.5, "Нет данных", ha="center", va="center",
                      color=GRAY_MID, fontsize=9)
        draw_programs_table(fig.add_subplot(gs_right[2]), program_rows)

        # Строка 1, cols 0–7: метка раздела А
        draw_section_label(
            fig.add_subplot(gs[1, :8]),
            "Общие результаты опроса по программам преподавателя",
        )

        # Строки 2–3, cols 0–7: 8 карточек (4 × 2 строки, по 2 col каждая)
        for idx, (title, value, accent) in enumerate(section_a_cards):
            draw_metric_card(
                fig.add_subplot(gs[2 + idx // 4, (idx % 4) * 2 : (idx % 4) * 2 + 2]),
                title, value, accent=accent,
            )

        # Строка 4, cols 0–7: метка раздела Б
        draw_section_label(
            fig.add_subplot(gs[4, :8]),
            "Персональные оценки студентами",
        )

        # Строка 5, cols 0–7: 4 карточки (по 2 col каждая)
        for col_i, (lbl, val, accent) in enumerate(b_cards):
            draw_stat_card(
                fig.add_subplot(gs[5, col_i * 2 : col_i * 2 + 2]),
                lbl, val, accent=accent,
            )

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)


def resolve_output_path(teacher_name: str, teacher_df: pd.DataFrame) -> Path:
    """output/teacher_pdfs/<Школа>/<Программа>/<Имя>.pdf"""
    fname = safe_filename(teacher_name) + ".pdf"
    group_cols = [c for c in ("school", "program") if c in teacher_df.columns]
    if group_cols:
        dominant = teacher_df.groupby(group_cols)["rating"].count().idxmax()
        if len(group_cols) == 1:
            dominant = (dominant,)
        parts = [safe_filename(str(v)) for v in dominant]
    else:
        parts = []
    path = OUTPUT_DIR.joinpath(*parts)
    path.mkdir(parents=True, exist_ok=True)
    return path / fname


print("Функции генерации определены.")

Функции генерации определены.


In [113]:
# ── Фильтры ────────────────────────────────────────────────────────────────
# Укажите значения ниже, затем запустите ячейку.

SEL_SCHOOLS  = ["МАРШ"]   # [] = все школы;  иначе: ["Школа A", "Школа B"]
SEL_PROGRAMS = []   # [] = все программы; иначе: ["MBA", "EMBA"]
MIN_N        = 3    # минимум ответов для включения преподавателя
ONLY_ONE     = "Александр Аляев"   # "" = все; или имя одного преподавателя точно как в CSV

# ── Фильтрация ─────────────────────────────────────────────────────────────
df = teachers_df.copy()
if SEL_SCHOOLS and "school" in df.columns:
    df = df[df["school"].isin(SEL_SCHOOLS)]
if SEL_PROGRAMS:
    df = df[df["program"].isin(SEL_PROGRAMS)]

counts = df.groupby("teacher")["rating"].count()
if ONLY_ONE:
    teachers_to_run = [ONLY_ONE] if ONLY_ONE in counts.index else []
    if not teachers_to_run:
        print(f"Преподаватель «{ONLY_ONE}» не найден в отфильтрованных данных.")
else:
    teachers_to_run = sorted(counts[counts >= MIN_N].index.tolist())

n_total = len(teachers_to_run)
print(f"Генерация: {n_total} PDF  |  Мин. ответов: {MIN_N}")
if SEL_SCHOOLS:  print(f"  Школы:     {SEL_SCHOOLS}")
if SEL_PROGRAMS: print(f"  Программы: {SEL_PROGRAMS}")

# ── Генерация ──────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ok, err = 0, 0

for i, name in enumerate(teachers_to_run, 1):
    t_df  = df[df["teacher"] == name].copy()
    fpath = resolve_output_path(name, t_df)
    try:
        generate_pdf_for_teacher(name, t_df, general_df, str(fpath))
        ok += 1
        if i % 25 == 0 or i == n_total:
            print(f"  [{i:>3}/{n_total}] {ok} готово, {err} ошибок")
    except Exception as e:
        err += 1
        print(f"  [{i:>3}/{n_total}] ✗ {name}: {e}")

print(f"\nГотово: {ok} PDF · {err} ошибок")
print(f"Папка:  {OUTPUT_DIR.resolve()}")

Генерация: 1 PDF  |  Мин. ответов: 3
  Школы:     ['МАРШ']


  [  1/1] 1 готово, 0 ошибок

Готово: 1 PDF · 0 ошибок
Папка:  /Users/matveibisler/Library/CloudStorage/Dropbox/UU/Code/SFQ 2025:26/output/teacher_pdfs
